# CODaN Feature Extraction
This notebook converts CODaN CSV image indexes into HSV value features. It runs `process_codan_csv` on the train, test, and val splits.

## Imports

In [1]:
from pathlib import Path
import pandas as pd
import cv2

## process_codan_csv

In [2]:
def process_codan_csv(csv_path, out_file=None):
    csv_path = Path(csv_path)
    repo_root = Path(__file__).resolve().parents[2] if '__file__' in globals() else Path.cwd()
    if not csv_path.is_absolute():
        csv_path = repo_root / csv_path

    if not csv_path.exists():
        raise FileNotFoundError(f'Index CSV not found: {csv_path}')

    if out_file:
        out_path = Path(out_file)
    else:
        out_path = csv_path.parent / 'image_values.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(csv_path, header=0)
    if df.shape[1] < 2:
        raise ValueError('Input CSV must have at least two columns: filename, label')
    if 'filename' in df.columns and 'class' in df.columns:
        df = df.rename(columns={'class': 'label'})[['filename', 'label']]
    else:
        df = df.iloc[:, :2]
        df.columns = ['filename', 'label']

    base_data_dir = repo_root / 'data' / 'CODaN'
    rows = []
    missing = 0
    for idx, row in df.iterrows():
        fname = str(row['filename'])
        label = row['label']
        fpath = Path(fname)
        candidates = []
        if fpath.is_absolute():
            candidates.append(fpath)
        else:
            candidates.append(csv_path.parent / fpath)
            candidates.append(base_data_dir / 'train' / fpath)
            candidates.append(base_data_dir / 'test' / fpath)
            candidates.append(base_data_dir / 'val' / fpath)
            candidates.append(base_data_dir / fpath)
            for p in base_data_dir.rglob(fpath.name):
                candidates.append(p)

        img_path = None
        for c in candidates:
            if c.exists() and c.is_file():
                img_path = c
                break

        if img_path is None:
            print(f'Warning: image not found for entry {fname} (skipping)')
            missing += 1
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            print(f'Warning: failed to read image: {img_path} (skipping)')
            missing += 1
            continue
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        avg_v = float(hsv[:, :, 2].mean()) / 255.0
        rows.append((fname, avg_v, int(label)))

    out_df = pd.DataFrame(rows, columns=['filename', 'avg_value', 'label'])
    out_df.to_csv(out_path, index=False)
    print(f'Wrote {len(out_df)} rows to {out_path} (skipped {missing} missing)')
    return out_df

## main function

In [21]:
def main(data_dir='../../data/CODaN', section='train'):
    print( "Tracing arguments..." )
    for name, value in locals().items():
        print( f"{name}: {value}" )
    print( "Tracing done." )  

    csv_path = data_dir + "/" + section + "/" + f'{section}.csv'
    out_path = data_dir + "/" + section + "/" + f'{section}_image_values.csv'
    print(f'Processing {section} split: {csv_path}')
    results.append(process_codan_csv(csv_path, out_file=out_path))
    
    return pd.concat(results, ignore_index=True)

## Run on train/test/val splits

In [23]:
results = main(data_dir='../../data/CODaN', section="train")
results.head()

Tracing arguments...
data_dir: ../../data/CODaN
section: train
Tracing done.
Processing train split: ../../data/CODaN/train/train.csv


AttributeError: 'DataFrame' object has no attribute 'append'